In [1]:
# 0. IMPORTS & LOAD DATA

import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data")

# Load historical segmentation table (output from Customer_segmentation.ipynb)
full = pd.read_pickle(DATA_DIR / "customer_segmentation_output_final.pkl")
full["fullVisitorId"] = full["fullVisitorId"].astype(str)

print("full shape:", full.shape)
print("Unique users in full:", full["fullVisitorId"].nunique())

# Load new segment prediction table
cus_predict = pd.read_csv(
    DATA_DIR / "customer_segmentation_prediction.csv",
    dtype={"fullVisitorId": str}
)

print("\ncus_predict shape:", cus_predict.shape)
print("Unique users in cus_predict:", cus_predict["fullVisitorId"].nunique())
print("\ncus_predict columns:", cus_predict.columns.tolist())

full shape: (1272948, 69)
Unique users in full: 1272948

cus_predict shape: (119207, 72)
Unique users in cus_predict: 119207

cus_predict columns: ['fullVisitorId', 'visitId_count', 'visitNumber_max', 'totals_hits_sum', 'funnel_depth', 'totals_pageviews_sum', 'totals_pageviews_mean', 'totals_timeOnSite_sum', 'totals_timeOnSite_mean', 'totals_bounces_mean', 'totals_sessionQualityDim_mean', 'totals_sessionQualityDim_max', 'totals_transactionRevenue_sum', 'totals_transactionRevenue_mean', 'totals_transactionRevenue_max', 'totals_transactions_sum', 'device_isMobile_mean', 'device_deviceCategory_enc_mean', 'device_deviceCategory_enc_last', 'geoNetwork_city_enc_mean', 'geoNetwork_city_enc_max', 'geoNetwork_country_enc_mean', 'geoNetwork_country_enc_max', 'geoNetwork_region_enc_mean', 'geoNetwork_metro_enc_mean', 'geoNetwork_subContinent_enc_mean', 'channelGrouping_enc_mean', 'channelGrouping_enc_max', 'channelGrouping_enc_last', 'trafficSource_medium_enc_mean', 'trafficSource_medium_enc_max'

In [2]:
# Check overlap between the two tables

hist_ids  = set(full["fullVisitorId"])
pred_ids  = set(cus_predict["fullVisitorId"])

common_ids     = hist_ids & pred_ids
only_in_hist   = hist_ids - pred_ids
only_in_pred   = pred_ids - hist_ids

print(f"Users in historical  : {len(hist_ids):>10,}")
print(f"Users in prediction  : {len(pred_ids):>10,}")
print(f"Users in BOTH (inner): {len(common_ids):>10,}")
print(f"Only in historical   : {len(only_in_hist):>10,}")
print(f"Only in prediction   : {len(only_in_pred):>10,}")

Users in historical  :  1,272,948
Users in prediction  :    119,207
Users in BOTH (inner):      1,571
Only in historical   :  1,271,377
Only in prediction   :    117,636


In [3]:
# MERGE SEGMENT / CLUSTER LABELS FROM BOTH TABLES
# Keep only visitors present in BOTH tables (inner join)

# 1. Extract cluster/segment columns from historical table

hist_cols = [
    "fullVisitorId",
    "Buyer_Flag",
    "Buyer_Segment",
    "Main_Segment",
    "Buyer_Cluster",
    "NonBuyer_Cluster",
    "Final_Cluster",
    "Final_Cluster_Name",
    "Objective",
    "Recommended_Strategy",
    "Avoid",
    "KPI",
    "Buyer_Strategy",
]

hist_cols = [c for c in hist_cols if c in full.columns]

hist_label_df = full[hist_cols].copy()

# Add hist_ prefix for easy comparison
hist_label_df = hist_label_df.rename(
    columns={
        c: f"hist_{c}"
        for c in hist_label_df.columns
        if c != "fullVisitorId"
    }
)

# 2. Extract cluster/segment columns from prediction table

pred_cols = [
    "fullVisitorId",
    "cluster",
    "cluster_label",
    "business_segment_4",
    "is_high_p",
    "is_high_clv",
]

pred_cols = [c for c in pred_cols if c in cus_predict.columns]

pred_label_df = cus_predict[pred_cols].copy()

# Add pred_ prefix for easy comparison
pred_label_df = pred_label_df.rename(
    columns={
        c: f"pred_{c}"
        for c in pred_label_df.columns
        if c != "fullVisitorId"
    }
)

# 3. Inner join on fullVisitorId
# Keep only users present in BOTH tables

cluster_compare_df = hist_label_df.merge(
    pred_label_df,
    on="fullVisitorId",
    how="inner"
)

print("Historical label df shape :", hist_label_df.shape)
print("Prediction label df shape :", pred_label_df.shape)
print("Merged compare df shape   :", cluster_compare_df.shape)
print(f"\nUsers filtered (inner)    : {cluster_compare_df['fullVisitorId'].nunique():,}")
print("Columns:", cluster_compare_df.columns.tolist())

cluster_compare_df.head()

Historical label df shape : (1272948, 13)
Prediction label df shape : (119207, 3)
Merged compare df shape   : (1571, 15)

Users filtered (inner)    : 1,571
Columns: ['fullVisitorId', 'hist_Buyer_Flag', 'hist_Buyer_Segment', 'hist_Main_Segment', 'hist_Buyer_Cluster', 'hist_NonBuyer_Cluster', 'hist_Final_Cluster', 'hist_Final_Cluster_Name', 'hist_Objective', 'hist_Recommended_Strategy', 'hist_Avoid', 'hist_KPI', 'hist_Buyer_Strategy', 'pred_cluster', 'pred_cluster_label']


,fullVisitorId,hist_Buyer_Flag,hist_Buyer_Segment,hist_Main_Segment,hist_Buyer_Cluster,hist_NonBuyer_Cluster,hist_Final_Cluster,hist_Final_Cluster_Name,hist_Objective,hist_Recommended_Strategy,hist_Avoid,hist_KPI,hist_Buyer_Strategy,pred_cluster,pred_cluster_label
0,1000035437777728789,0,Non-Buyer,Non-Buyer,NaN,4.0,NonBuyer_4,Warm Prospects,Nurture interested non-buyers and move them cl...,"Product education, comparison guides, personal...",Avoid heavy discounting before purchase intent...,"Product view depth, return visit rate, add-to-...",Activation / Conversion / First purchase,0,"Ưu tiên thấp: P thấp, CLV thấp"
1,100104898171048438,0,Non-Buyer,Non-Buyer,NaN,4.0,NonBuyer_4,Warm Prospects,Nurture interested non-buyers and move them cl...,"Product education, comparison guides, personal...",Avoid heavy discounting before purchase intent...,"Product view depth, return visit rate, add-to-...",Activation / Conversion / First purchase,0,"Ưu tiên thấp: P thấp, CLV thấp"
2,1010162378635762824,0,Non-Buyer,Non-Buyer,NaN,4.0,NonBuyer_4,Warm Prospects,Nurture interested non-buyers and move them cl...,"Product education, comparison guides, personal...",Avoid heavy discounting before purchase intent...,"Product view depth, return visit rate, add-to-...",Activation / Conversion / First purchase,0,"Ưu tiên thấp: P thấp, CLV thấp"
3,1016796741961860489,0,Non-Buyer,Non-Buyer,NaN,3.0,NonBuyer_3,Hot Prospects,Convert high-intent non-buyers into first-time...,"Prioritized retargeting, first-purchase incent...",Avoid generic awareness campaigns because this...,"First purchase conversion rate, add-to-cart ra...",Activation / Conversion / First purchase,1,"Ngôi sao: P cao nhất, CLV cao"
4,1017158871483575113,1,Buyer,Buyer,2.0,NaN,Buyer_2,Recent Quality Buyers,Convert recent quality buyers into repeat buyers,"Post-purchase onboarding, second-purchase ince...",Avoid treating them as loyal buyers too early,"Second purchase rate, time to next purchase, r...",Retention / Upsell / Cross-sell,0,"Ưu tiên thấp: P thấp, CLV thấp"


In [4]:
cus_predict.info()

<class 'pandas.DataFrame'>
RangeIndex: 119207 entries, 0 to 119206
Data columns (total 72 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   fullVisitorId                     119207 non-null  str    
 1   visitId_count                     119207 non-null  int64  
 2   visitNumber_max                   119207 non-null  int64  
 3   totals_hits_sum                   119207 non-null  int64  
 4   funnel_depth                      119207 non-null  float64
 5   totals_pageviews_sum              119207 non-null  float64
 6   totals_pageviews_mean             119207 non-null  float64
 7   totals_timeOnSite_sum             119207 non-null  float64
 8   totals_timeOnSite_mean            119207 non-null  float64
 9   totals_bounces_mean               119207 non-null  float64
 10  totals_sessionQualityDim_mean     119207 non-null  float64
 11  totals_sessionQualityDim_max      119207 non-null  int64  
 12 

In [5]:
import numpy as np

# ── 1. Tag New vs Returning
cus_predict["visitor_type"] = cus_predict["fullVisitorId"].apply(
    lambda x: "Returning" if x in hist_ids else "New"
)

print("Visitor type distribution:")
print(cus_predict["visitor_type"].value_counts())
print(f"New rate: {(cus_predict['visitor_type'] == 'New').mean():.1%}")


# ── 2. Expected Revenue (log scale → USD)
# e_revenue is log scale → expm1() to convert to USD
# expected_revenue_usd = expm1(e_revenue) × p_buy
cus_predict["expected_revenue_usd"] = (
    np.expm1(cus_predict["e_revenue"]) * cus_predict["p_buy"] / 1e6
)

print("\nExpected revenue (USD) sanity check:")
print(cus_predict.groupby("cluster_label")["expected_revenue_usd"].describe().round(4))


# ── 3. Short cluster labels (for chart labels)
cus_predict["cluster_short"] = cus_predict["cluster_label"].map({
    "Stars":   "Stars",
    "Low Priority":  "Low Priority",
}).fillna(cus_predict["cluster_label"])


# # ── 4. Revenue bucket (for distribution chart)
# cus_predict["revenue_bucket"] = pd.cut(
#     cus_predict["expected_revenue_usd"],
#     bins=[0, 0.01, 0.1, 1, 10, 100, float("inf")],
#     labels=["<$0.01", "$0.01–0.1", "$0.1–1", "$1–10", "$10–100", ">$100"],
#     include_lowest=True,
# )


# ── 5. Quick check before export ──
print("\nCluster summary:")
print(
    cus_predict.groupby("cluster_label").agg(
        n_users              =("p_buy",                "count"),
        mean_p_buy           =("p_buy",                "mean"),
        mean_expected_rev    =("expected_revenue_usd", "mean"),
        total_expected_rev   =("expected_revenue_usd", "sum"),
        pct_new              =("visitor_type",         lambda x: (x == "New").mean()),
        pct_has_purchased    =("has_purchased",        "mean"),
        pct_hibernating      =("hibernation_flag",     "mean"),
    ).round(4)
)


# ── 6. Export CSV for Dashboard 1 ──
export_cols = [
    # Identity
    "fullVisitorId",

    # Segment
    "cluster",
    "cluster_label",
    "cluster_short",
    "visitor_type",

    # Prediction outputs
    "p_buy",
    "e_revenue",
    "expected_revenue_usd",
    "final_score",
    "revenue_bucket",

    # Behavioral features
    "visitId_count",
    "recency_days",
    "lifetime_days",
    "has_purchased",
    "hibernation_flag",
    "p_alive",
    "expected_purchases_92d",
    "expected_monetary",

    # Engagement
    "totals_pageviews_mean",
    "totals_timeOnSite_mean",
    "totals_bounces_mean",
    "vel_sessions",
    "diversity_score",

    # Device
    "device_isMobile_mean",
]

export_cols = [c for c in export_cols if c in cus_predict.columns]

df_dashboard1 = (
    cus_predict[export_cols]
    .sort_values("final_score", ascending=False)
    .reset_index(drop=True)
)

# output_path = DATA_DIR / "dashboard1_users.csv"
# df_dashboard1.to_csv(output_path, index=False)

print(f"\n✅ Exported: {df_dashboard1.shape}")
# print(f"   → {output_path}")
# print(f"\nCols exported ({len(export_cols)}):")
# print(export_cols)

Visitor type distribution:
visitor_type
New          117636
Returning      1571
Name: count, dtype: int64
New rate: 98.7%

Expected revenue (USD) sanity check:
                                   count    mean     std    min     25%  \
cluster_label                                                             
Ngôi sao: P cao nhất, CLV cao     2462.0  2.4225  2.1362  0.835  1.3734   
Ưu tiên thấp: P thấp, CLV thấp  116745.0  0.0504  0.1259  0.000  0.0011   

                                   50%     75%      max  
cluster_label                                            
Ngôi sao: P cao nhất, CLV cao   1.7723  2.9812  49.3307  
Ưu tiên thấp: P thấp, CLV thấp  0.0099  0.0375   2.2625  

Cluster summary:
                                n_users  mean_p_buy  mean_expected_rev  \
cluster_label                                                            
Ngôi sao: P cao nhất, CLV cao      2462      0.0402             2.4225   
Ưu tiên thấp: P thấp, CLV thấp   116745      0.0010             0.0

In [6]:
stars = cus_predict[cus_predict["cluster_label"].str.contains("Stars")]

print("Expected revenue (USD) after dividing by 1e6:")
print(stars["expected_revenue_usd"].describe().round(4))

print(f"\nMean expected revenue per Star user: ${stars['expected_revenue_usd'].mean():.4f}")
print(f"Total expected revenue (Stars):      ${stars['expected_revenue_usd'].sum():.2f}")
print(f"Total expected revenue (All):        ${cus_predict['expected_revenue_usd'].sum():.2f}")

Expected revenue (USD) sau khi chia 1e6:
count    2462.0000
mean        2.4225
std         2.1362
min         0.8350
25%         1.3734
50%         1.7723
75%         2.9812
max        49.3307
Name: expected_revenue_usd, dtype: float64

Mean expected revenue per Star user: $2.4225
Total expected revenue (Stars):      $5964.25
Total expected revenue (All):        $11845.25


In [7]:
df_dashboard1.info()

<class 'pandas.DataFrame'>
RangeIndex: 119207 entries, 0 to 119206
Data columns (total 23 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   fullVisitorId           119207 non-null  str    
 1   cluster                 119207 non-null  int64  
 2   cluster_label           119207 non-null  str    
 3   cluster_short           119207 non-null  str    
 4   visitor_type            119207 non-null  str    
 5   p_buy                   119207 non-null  float64
 6   e_revenue               119207 non-null  float64
 7   expected_revenue_usd    119207 non-null  float64
 8   final_score             119207 non-null  float64
 9   visitId_count           119207 non-null  int64  
 10  recency_days            119207 non-null  int64  
 11  lifetime_days           119207 non-null  int64  
 12  has_purchased           119207 non-null  int64  
 13  hibernation_flag        119207 non-null  int64  
 14  p_alive                 119207 

In [8]:
df_dashboard1['visitor_type'].value_counts()

visitor_type
New          117636
Returning      1571
Name: count, dtype: int64

In [9]:
output_path = '../data/clv.csv'
df_dashboard1.to_csv(output_path, index=False)

In [10]:
df_dashboard1['expected_revenue_usd'].describe()

count    119207.000000
mean          0.099367
std           0.472828
min           0.000014
25%           0.001182
50%           0.010554
75%           0.039494
max          49.330663
Name: expected_revenue_usd, dtype: float64

In [ ]:
output_path = '../data/clv.csv'
df_dashboard1.to_csv(output_path, index=False)

In [ ]:
df_dashboard1['cluster_short'].value_counts()

cluster_short
Low Priority    116745
Stars             2462
Name: count, dtype: int64

In [ ]:
from IPython.display import display

clv_by_cluster = (
    df_dashboard1
    .groupby("cluster_short", observed=True)
    .agg(
        n_users          = ("fullVisitorId",        "count"),
        total_clv        = ("expected_revenue_usd",  "sum"),
        mean_clv         = ("expected_revenue_usd",  "mean"),
        median_clv       = ("expected_revenue_usd",  "median"),
    )
    .sort_values("total_clv", ascending=False)
    .reset_index()
)

# % revenue contribution
clv_by_cluster["pct_revenue"] = (
    clv_by_cluster["total_clv"] / clv_by_cluster["total_clv"].sum() * 100
).round(2)

display(clv_by_cluster)
print(f"\nTotal CLV (all): ${clv_by_cluster['total_clv'].sum():,.2f}")


,cluster_short,n_users,total_clv,mean_clv,median_clv,pct_revenue
0,Stars,2462,5964.248559,2.422522,1.772348,50.35
1,Low Priority,116745,5880.998892,0.050375,0.009895,49.65



Total CLV toàn bộ: $11,845.25


In [ ]:
clv_by_cluster["pct_revenue"] = (
    clv_by_cluster["total_clv"] / clv_by_cluster["total_clv"].sum() * 100
).round(2)

In [ ]:
total_revenue = df_dashboard1["expected_revenue_usd"].sum()
print(f"Total expected revenue (all users): ${total_revenue:,.2f}")


Total expected revenue (all users): $11,845.25


In [ ]:
# Check cluster names first
print(df_dashboard1["cluster_short"].unique())

# Then filter by the correct name (e.g. "Stars")
avg_clv_star = df_dashboard1.loc[
    df_dashboard1["cluster_short"] == "Stars",
    "expected_revenue_usd"
].mean()

print(f"Average CLV (Stars): ${avg_clv_star:,.4f}")


<StringArray>
['Stars', 'Low Priority']
Length: 2, dtype: str
Average CLV (Stars): $2.4225
